# 1주차 보충 · 낯선 문제를 읽고 예측까지

진품임을 아직 모르는 그림과 복제품이라고 확인한 그림을 같은 무리에 넣어도 될까요? 새 문제를 받으면 먼저 **한 행이 누구인지, 어느 순간 무엇을 아는지, 답 한 줄에 무엇을 쓰는지** 정해 봅시다.

- 처음이라면 1–3절에서 감정 기록을 읽고 훈련 자료로 짧은 분류 규칙을 배웁니다. 표에서 한 행을 고르고 빈칸을 뺀 평균을 구할 수 있으면 시작할 수 있습니다. [표와 평균 다시 보기](week01_integrated.ipynb)
- 통합 교재에서 제출 순서를 확인하러 왔다면 [4절의 세 작품](#pipeline-submit)만 풀고 돌아가세요. 이 예제는 앞의 코드를 실행하지 않아도 됩니다.
- 모델을 써 본 적이 있다면 [5절의 새 기기 문제](#pipeline-device)에서 바로 자기 답을 만들 수 있습니다. 실제 ZIP의 내부가 궁금할 때는 맨 뒤의 [파일 실험](#pipeline-files)을 보세요.

코드 실습에는 Python, pandas, NumPy와 scikit-learn을 사용합니다. 표의 숫자는 이 연습을 위해 만든 값입니다.


<a id="pipeline-contract"></a>

## 1. 문제지에서 한 행과 답 한 줄 찾기

국제 대회의 *Antique Painting Authentication*에서 그림 한 점의 디지털 측정값 다섯 개가 주어집니다. 훈련 파일의 여섯째 값은 1이면 진품, −1이면 복제품, 0이면 감정 결과 미상입니다. 문제는 미상인 그림의 측정값까지 활용해 진품과 복제품을 분류하라고 합니다. 다음 세 줄을 문제지처럼 읽어 보세요. [IOAI 2025 개인전 원문 1–5절](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Antique/Antique.ipynb)

| 받은 조건 | 무엇을 결정하나요? |
|---|---|
| 훈련·검증·테스트에는 각각 500점이 있다. 대회 중에는 훈련 자료를 직접 볼 수 있다. 채점 환경에서 실행되는 코드가 검증·테스트의 측정값을 읽는다. | 참가자가 훈련할 때 볼 수 있는 것과 채점 시 코드가 받는 것을 나눠 보세요. |
| 제출할 <code>submission.ipynb</code>는 <code>submission.zip</code> 안에 검증용 <code>submissionA.csv</code>와 테스트용 <code>submissionB.csv</code>를 만든다. 두 파일 모두 헤더 없이 그림마다 −1 또는 1 하나를 쓴다. | 입력 세 번째 그림의 예측은 어느 줄에 올까요? 훈련에서 본 0도 답으로 쓸 수 있을까요? |
| 각 파일에서 올바르게 분류한 작품 수를 그 파일의 전체 작품 수로 나눈다. | 채점에서 한 번을 세는 대상은 측정값 하나일까요, 그림 한 점일까요? |

검증·테스트 입력에는 라벨 열이 없습니다. 해당 파일의 세 번째 그림에 대한 예측 한 개가 제출 파일의 세 번째 줄에 옵니다. 훈련의 ‘미상’은 감정 결과를 아직 모른다는 뜻입니다.

**다른 문제지를 받으면.** *Potato*에서는 심판이 두 단어 중 어느 쪽이 숨은 단어에 더 가까운지 알려 주고, 참가자의 코드는 다음 단어 하나를 제안합니다. 훈련 묶음은 없고 자가 채점용 공개 게임의 정답만 주어집니다. 순위에 쓰는 게임의 정답은 숨겨져 있습니다. 친구가 “두 문제 모두 공개 정답 행으로 분류기부터 학습하자”고 합니다. Antique에서는 무엇으로 학습할 수 있고, Potato에서는 어떤 자료로 제안 방법을 점검할 수 있을까요? 두 문제지에서 각각 근거를 한 가지씩 찾으세요. [IOAI 2026 Potato 원문의 Task·Dataset·Output](https://github.com/IOAI-official/IOAI-2026/blob/e1963801aa4cf33a5a03520b39fa6620a7894c54/Individual-Contest/3_Potato/statement.md)


### 같은 질문을 작은 표에 옮기기

아래 수집품은 가상의 두 점수 <code>texture</code>와 <code>color</code>를 감정 전에 같은 척도로 측정합니다. <code>row_id</code>는 수집품에 붙은 번호표입니다. 감정 후 기록하는 <code>label</code>은 원문과 같이 −1, 1, 0을 사용합니다.

이번 규칙은 **정답이 확인된 훈련 수집품**으로만 두 무리의 중심을 배웁니다. 어느 열을 숫자 계산에 쓰고 어느 열을 나중에 결과를 찾는 데 쓸까요? 정답이 없는 T5의 측정값까지 활용하라는 원문의 요구는 이 규칙에서 어디가 남을까요?


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import Pipeline

feature_cols = ["texture", "color"]
training = pd.DataFrame([
    ("T1", 1, 1, -1), ("T2", 3, np.nan, -1), ("T3", 7, 3, 1),
    ("T4", 9, 5, 1), ("T5", 6, 3, 0),
], columns=["row_id", "texture", "color", "label"])
known_train = training[training["label"].isin([-1, 1])].copy()
display(training)
print("이 예에서 정답을 기준으로 묶을 수 있는 작품:", known_train["row_id"].tolist())


,row_id,texture,color,label
0,T1,1,1.0,-1
1,T2,3,NaN,-1
2,T3,7,3.0,1
3,T4,9,5.0,1
4,T5,6,3.0,0


이 예에서 정답을 기준으로 묶을 수 있는 작품: ['T1', 'T2', 'T3', 'T4']


## 2. 모델이 실제로 배운 두 점 찾기

<code>fit</code>은 훈련 자료로부터 다음 예측에 쓸 값을 구합니다. T2의 색 점수가 비어서, 알려진 훈련 작품 중 색을 관측한 세 점의 평균 (1+3+5)÷3=3으로 채웁니다. 조직감 점수의 평균은 (1+3+7+9)÷4=5입니다. T5의 정답은 미상이므로 이 예에서는 대체값을 구하거나 어느 무리의 중심을 정할 때도 쓰지 않습니다.

같은 라벨끼리 모아 열마다 평균을 내면 그 무리의 **중심**입니다. −1 무리의 두 점 [1,1]과 [3,3]을 대표하는 중심은 [2,2]입니다. +1 무리의 두 점에서 나머지 중심을 직접 구해 보세요.

새 작품 [6,3]과 −1 중심 사이의 **제곱거리**는 (6−2)²+(3−2)²=17입니다. +1 중심까지도 같은 계산을 하고 두 수 중 작은 쪽의 라벨을 적으세요. 제곱근을 취해도 두 거리의 크기 순서가 같아 여기서는 합까지만 계산합니다. 두 특성을 같은 척도로 둔 이유도 생각해 보세요.


In [2]:
def new_model():
    return Pipeline([("fill", SimpleImputer(strategy="mean")),
                     ("classifier", NearestCentroid())])

model = new_model().fit(known_train[feature_cols], known_train["label"])
fill_values = model.named_steps["fill"].statistics_
centers = model.named_steps["classifier"].centroids_
class_names = model.named_steps["classifier"].classes_
print("훈련에서 배운 대체값:", fill_values)
for label, center in zip(class_names, centers):
    distance_squared = np.sum((np.array([6, 3]) - center) ** 2)
    print(f"라벨 {label}의 중심 {center}, 새 작품까지 제곱거리 {distance_squared:g}")


훈련에서 배운 대체값: [5. 3.]
라벨 -1의 중심 [2. 2.], 새 작품까지 제곱거리 17
라벨 1의 중심 [8. 4.], 새 작품까지 제곱거리 5


### 훈련값 하나를 바꿔 보기

T4의 조직감 점수를 9에서 11로 바꾼 사본에 새 모델을 학습하면 어느 중심이 어디로 움직일까요? 바뀐 중심에 가까웠던 경계 부근의 새 점 [5.25,3]도 같은 라벨을 받을지 먼저 예상하세요. 처음 학습한 모델은 그대로 두고 두 모델에 같은 점을 보여 줍니다.


In [3]:
changed_train = known_train.copy()
changed_train.loc[changed_train["row_id"] == "T4", "texture"] = 11
changed_model = new_model().fit(changed_train[feature_cols], changed_train["label"])
print("원래 1의 중심:", model.named_steps["classifier"].centroids_[1])
print("훈련값을 바꾼 1의 중심:",
      changed_model.named_steps["classifier"].centroids_[1])
print("원래 모델 대체값 유지:", model.named_steps["fill"].statistics_)
probe = pd.DataFrame([[5.25, 3]], columns=feature_cols)
original_probe = int(model.predict(probe)[0])
changed_probe = int(changed_model.predict(probe)[0])
print("같은 새 입력의 원래 / 훈련 사본 변경 모델 예측:",
      original_probe, "/", changed_probe)
assert np.allclose(changed_model.named_steps["classifier"].centroids_[1], [9, 4])
assert np.allclose(model.named_steps["classifier"].centroids_[1], [8, 4])
assert (original_probe, changed_probe) == (1, -1)


원래 1의 중심: [8. 4.]
훈련값을 바꾼 1의 중심: [9. 4.]
원래 모델 대체값 유지: [5. 3.]
같은 새 입력의 원래 / 훈련 사본 변경 모델 예측: 1 / -1


## 3. 예측을 저장한 뒤 검증 정답 열기

검증용 V1~V4는 훈련 작품과 겹치지 않습니다. V3의 점수는 [6,3]이고 V4에는 조직감 점수가 없습니다. V4에 무슨 값을 넣을지, 그 값을 **어느 작품들에서** 배웠는지 말해 보세요. 새 모델을 학습하면 지금 점검하려던 규칙이 달라집니다.


In [4]:
validation = pd.DataFrame([
    ("V1", 2, 3), ("V2", 8, 3), ("V3", 6, 3), ("V4", np.nan, 5),
], columns=["row_id", "texture", "color"])
val_predictions = model.predict(validation[feature_cols])
worked_first = validation[["row_id"]].copy()
worked_first["prediction"] = val_predictions
display(worked_first)


,row_id,prediction
0,V1,-1
1,V2,1
2,V3,1
3,V4,1


이제 네 작품의 감정 결과를 엽니다. 감정소에서 돌려준 줄은 입력 순서와 다를 수 있습니다. **같은 ID의 첫 예측과 감정 결과가 맞는지** 비교하세요. 4개 중 몇 개가 맞았고 어느 작품에서 틀렸나요?


In [5]:
worked_validation_key = pd.DataFrame([
    ("V4", 1), ("V1", -1), ("V3", -1), ("V2", 1),
], columns=["row_id", "label"])
val_view = worked_first.merge(worked_validation_key, on="row_id", validate="one_to_one")
assert len(val_view) == len(worked_first) == len(worked_validation_key)
val_view["correct"] = val_view["prediction"] == val_view["label"]
accuracy = val_view["correct"].sum() / len(val_view)
display(val_view)
print("맞은 작품 / 평가한 작품:", val_view["correct"].sum(), "/", len(val_view))


,row_id,prediction,label,correct
0,V1,-1,-1,True
1,V2,1,1,True
2,V3,1,-1,False
3,V4,1,1,True


맞은 작품 / 평가한 작품: 3 / 4


3/4은 지금 본 네 작품의 정확도입니다. V3에서는 계산한 예측과 실제 감정 결과가 달랐습니다. 검증으로 방법을 고를 수 있지만, 같은 정답을 보면서 계속 고치면 그 네 작품에만 맞을 수도 있습니다. 마지막 성능을 확인할 새 테스트의 정답은 여기서 제공하지 않습니다.

훈련 T4를 바꾼 앞 실험과 이번 실험의 차이를 보세요. 아래에서는 검증 행만 바꾸고 이미 배운 모델의 <code>predict</code>를 호출합니다. 대체값이나 중심도 바뀔까요?


In [6]:
before_fill = model.named_steps["fill"].statistics_.copy()
before_centers = model.named_steps["classifier"].centroids_.copy()
changed_validation = validation.copy()
changed_validation.loc[changed_validation["row_id"] == "V1", "texture"] = 2000
_ = model.predict(changed_validation[feature_cols])
print("검증을 바꾼 뒤 대체값:", model.named_steps["fill"].statistics_)
print("검증을 바꾼 뒤 중심:\n", model.named_steps["classifier"].centroids_)
assert np.array_equal(model.named_steps["fill"].statistics_, before_fill)
assert np.array_equal(model.named_steps["classifier"].centroids_, before_centers)


검증을 바꾼 뒤 대체값: [5. 3.]
검증을 바꾼 뒤 중심:
 [[2. 2.]
 [8. 4.]]


<a id="pipeline-submit"></a>

## 4. 세 작품의 답을 어느 줄에 적을까?

예측을 만든 사람이 다음 표를 건넸습니다. **이 표만으로 풀 수 있습니다.** 이 작은 문제에서는 담당자가 요구하는 순서가 입력과 다릅니다.

| 예측할 때의 입력 순서 | 작품 ID | 모델의 예측 |
|---:|---|---:|
| 1 | A08 | 1 |
| 2 | A02 | −1 |
| 3 | A05 | −1 |

담당자가 적어 준 순서는 **A05, A08, A02**입니다. 헤더 없는 세 줄에 쓸 값을 위에서 아래로 적으세요. 동료가 입력 표의 마지막 열을 그대로 복사했습니다. 줄 수도, 허용된 값도 맞는데 어느 작품에 다른 예측이 붙을까요?


In [7]:
predicted_rows = [("A08", 1), ("A02", -1), ("A05", -1)]
requested_rows = ["A05", "A08", "A02"]
by_id = dict(predicted_rows)  # 이 표에서는 ID마다 예측이 하나입니다.
for row_id in requested_rows:
    print(row_id, "→", by_id[row_id])


A05 → -1
A08 → 1
A02 → -1


요구 순서에서는 −1, 1, −1입니다. 원래 값 1, −1, −1을 그대로 보내면 A05와 A08의 자리에 다른 값이 갑니다. 이것은 **저장할 때 고른 작품이 틀린 것**입니다. A의 참 라벨은 모르므로 이 비교를 모델의 정확도라고 부를 수 없습니다.

실제 Antique에는 별도 요구 ID 목록이나 출력 ID 열이 없습니다. 원래 입력 파일 순서로 예측을 쓰므로, 읽은 행을 섞었다면 읽을 때 보관한 자리를 따라 되돌립니다. 이름순으로 다시 정렬하라는 뜻도 아닙니다.

통합 교재에서 왔다면 ‘질의마다 무엇을 한 칸에 쓰는지’를 다시 답해 보세요. [통합 교재의 점수표로 돌아가기](week01_integrated.ipynb#week01-match)


<a id="pipeline-device"></a>

## 5. 혼자 풀기 · 기기 점검 의뢰서

**정비소의 부탁.** 기기 한 대가 들어올 때 직원은 번호표 <code>item</code>을 붙입니다. 센서는 그 자리에서 진동(<code>vibration</code>), 소리(<code>sound</code>), 열(<code>heat</code>)을 같은 척도의 점수로 기록합니다. 이때 점검이 필요할지 알려 주세요. 예전 장부에서 <code>label</code>은 다음 날 기술자가 확인한 결과로 0=정상, 1=점검, −1=미확인입니다. <code>after_service</code>에는 그 뒤에야 정비 영수증 번호가 들어갑니다. 미확인 기기는 지금 배우는 지도학습의 두 무리에서 제외합니다. 센서 한 칸이 비면 **정답이 확인된 훈련 기기**에서 그 센서가 기록된 값들의 평균을 씁니다.

정비소는 새 기기를 **받은 순서대로** 알려 달라고 했습니다. 파일 이름은 <code>audit.csv</code>이고, 첫 줄은 <code>item,signal</code>입니다. 나머지 줄에는 기기 번호표와 0 또는 1을 씁니다. 검증에서는 기술자가 확인한 기기 한 대마다 예측 한 번을 셉니다.

1. 완성 예제를 덮고, 담당자에게 설명하듯 **한 행의 뜻, 예측할 순간, 쓸 열과 제외할 열, 낼 답 한 줄**을 적으세요. 예전 장부에서 영수증 번호로 정답을 잘 맞히더라도 왜 새 기기에서는 그 방법을 쓸 수 없을까요?
2. 아래 표의 열 순서를 살펴보세요. 훈련에 사용할 ID와 빼는 ID를 고르고 대체 평균을 손계산하세요. 원하는 분류기를 이 훈련 자료에 새로 학습하고, 모델이 배운 값 한 가지를 손계산이나 원래 자료와 비교하세요.
3. 검증 정답을 열기 전에 세 기기의 첫 예측을 <code>first_device_predictions</code>라는 표에 <code>item,prediction</code> 두 열로 기록하세요. 새 기기의 <code>audit.csv</code>는 직접 쓰고 다시 열어 열 이름, 줄 수, ID 순서와 **기기별 예측값**이 맞는지 확인하세요. 새 기기의 **행 순서를 한 번 더 바꿔** 파일을 만들면 어느 줄이 움직이고 어느 값이 같은 기기에 남아야 할까요?

이 절의 데이터 셀부터 실행해도 됩니다. 분류기를 쓰려면 필요한 라이브러리를 직접 불러오세요.


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

exercise_train = pd.DataFrame([
    ("R1",  2,     0, 101, 0, 0), ("R2", np.nan, 0, 102, 2, 2),
    ("R3",  4,     1, 803, 2, 6), ("R4",      6, 1, 804, 4, 8),
    ("R5",  5,    -1, np.nan, 2, 4),
], columns=["item", "sound", "label", "after_service", "heat", "vibration"])
exercise_validation = pd.DataFrame([
    (1, "V1", 2, 3), (3, "V2", 6, 5), (1, "V3", 5, 4),
], columns=["heat", "item", "vibration", "sound"])
exercise_input = pd.DataFrame([
    (5, 7, "J3", 3), (3, 1, "J1", 1), (np.nan, 2, "J2", 1),
], columns=["sound", "vibration", "item", "heat"])

# 데이터 셀을 다시 실행해도 이미 기록한 첫 예측을 지우지 않습니다.
if "first_device_predictions" not in globals():
    first_device_predictions = None
if "recorded_device_predictions" not in globals():
    recorded_device_predictions = None
device_key_path = Path("data/pipeline_device_validation.csv")
print("예전 장부")
display(exercise_train)
print("검증 입력")
display(exercise_validation)
print("정비소에서 새로 받은 순서")
display(exercise_input)


예전 장부


,item,sound,label,after_service,heat,vibration
0,R1,2.0,0,101.0,0,0
1,R2,NaN,0,102.0,2,2
2,R3,4.0,1,803.0,2,6
3,R4,6.0,1,804.0,4,8
4,R5,5.0,-1,NaN,2,4


검증 입력


,heat,item,vibration,sound
0,1,V1,2,3
1,3,V2,6,5
2,1,V3,5,4


정비소에서 새로 받은 순서


,sound,vibration,item,heat
0,5.0,7,J3,3
1,3.0,1,J1,1
2,NaN,2,J2,1


내가 읽은 예측 시점·열·미상 규칙과 출력 한 줄의 뜻은 …

훈련 ID와 손계산, 선택한 모델과 그 모델이 배운 값은 …

<code>audit.csv</code>를 다시 읽고, 입력 순서를 바꾸어 대조한 두 기기의 번호표와 값은 …


In [9]:
# 새 명세에 맞는 학습, 첫 검증 예측, audit.csv 작성과 확인을 직접 적으세요.


### 첫 예측을 정답과 대조하기

첫 예측의 ID와 값이 모두 정해졌으면 아래 확인 셀을 실행하세요. 파일이 따로 있다면 <code>device_key_path</code>에 그 경로를 넣으세요. 자습할 때에는 첫 예측을 기록한 뒤에 [검증용 정답 CSV](data/pipeline_device_validation.csv)를 내려받아 <code>data</code> 폴더에 놓을 수 있습니다. 이 파일은 공개되어 있습니다. 이미 정답을 보았다면 받은 도움에 그렇게 적으세요. 정답 파일의 줄 순서는 달라도 됩니다. 새 기기 J의 정답은 주어지지 않습니다.


In [10]:
def device_values_by_item(table, value_column, expected_items):
    if not isinstance(table, pd.DataFrame) or table.columns.tolist() != ["item", value_column]:
        raise ValueError(f"필요한 두 열: item, {value_column}")
    expected_items = list(expected_items)
    received_items = table["item"].tolist()
    for items in (expected_items, received_items):
        if not all(isinstance(item, str) and item.strip() for item in items):
            raise ValueError("비어 있지 않은 기기 번호표가 필요합니다.")
        if len(items) != len(set(items)):
            raise ValueError("같은 번호표가 두 번 있습니다.")
    if set(received_items) != set(expected_items):
        raise ValueError("검증 기기의 번호표를 빠짐없이 한 번씩 써 주세요.")
    values = table.set_index("item")[value_column]
    for value in values:
        integer = isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_))
        if not integer or value not in (0, 1):
            raise ValueError("정수 0 또는 1을 써 주세요.")
    return tuple((item, int(values.at[item])) for item in expected_items)

device_comparison = None
if first_device_predictions is None:
    print("아직 첫 예측표가 없습니다. 위 답안에서 세 기기의 예측을 만들면 정답을 대조할 수 있습니다.")
else:
    first_rows = device_values_by_item(first_device_predictions, "prediction", exercise_validation["item"])
    if recorded_device_predictions is None:
        recorded_device_predictions = first_rows
    elif dict(first_rows) != dict(recorded_device_predictions):
        raise ValueError("보관한 첫 예측과 값이 다릅니다. 바꾼 예측은 별도로 비교하세요.")
    if Path(device_key_path).is_file():
        key_rows = pd.read_csv(device_key_path, dtype={"item": str})
        key_by_item = dict(device_values_by_item(key_rows, "label", exercise_validation["item"]))
        device_comparison = pd.DataFrame(recorded_device_predictions, columns=["item", "prediction"])
        device_comparison["label"] = device_comparison["item"].map(key_by_item)
        display(device_comparison)
    else:
        print("첫 예측을 보관했습니다. 정답 파일을 받은 뒤 device_key_path를 확인하고 다시 실행하세요.")


아직 첫 예측표가 없습니다. 위 답안에서 세 기기의 예측을 만들면 정답을 대조할 수 있습니다.


대조한 표에서 **맞은 기기 수 / 평가한 기기 수**와 첫 예측이 어긋난 ID를 직접 적으세요. 모두 맞았다면 그렇게 기록하세요. 이 표를 보고 설정을 고쳤다면 어느 결과가 첫 시도였는지도 남기세요. 센서 값 한 칸이 빈 경우와 열 자체가 없어진 경우, 또는 번호표가 중복된 경우에 같은 코드를 그대로 써도 될까요?

처음 생각과 달라진 점 / 받은 도움(없음·힌트·예제·설명) …

[통합 교재의 내 결과 확인으로 돌아가기](week01_integrated.ipynb#week01-check) · [2주차 분할·전처리·평가](../week02/week02_prediction_evaluation.ipynb)


In [11]:
# 첫 예측을 대조하여 맞은 개수, 전체 개수, 그리고 틀린 기기 번호표를 구하세요.


<a id="pipeline-files"></a>

## 더 해 보기 · A/B 파일을 실제로 열어 보기

4절의 A 표를 모델의 예측에서 직접 만들고 B 표도 하나 더 만들어 봅니다. 이 실험을 하려면 1–3절의 예제 코드를 먼저 실행합니다. 새 기기 문제와 다른 약속을 사용합니다. **각 CSV에는 ID나 헤더 없이 −1 또는 1만** 적습니다.

아래 도구는 한 셀의 결측에는 훈련 평균을 쓰고, 필요한 열 전체가 없거나 번호표가 빠지거나 겹치면 멈춥니다. A 입력의 열 순서를 바꾸거나 나중 감정 열을 덧붙여도 모델이 사용하는 열 이름은 그대로입니다.


In [12]:
def is_missing_id(value):
    if not pd.api.types.is_scalar(value):
        raise ValueError("ID는 하나의 값이어야 함")
    return bool(pd.isna(value)) or (isinstance(value, str) and not value.strip())

def predict_pairs(fitted_model, frame):
    required = ["row_id"] + feature_cols
    missing = [name for name in required if name not in frame.columns]
    if missing:
        raise ValueError("필수 열 없음: " + ", ".join(missing))
    if frame["row_id"].map(is_missing_id).any() or frame["row_id"].duplicated().any():
        raise ValueError("row_id 결측 또는 중복")
    values = frame[feature_cols].apply(pd.to_numeric, errors="raise")
    if np.isinf(values.to_numpy(dtype=float)).any():
        raise ValueError("특성에 무한값은 허용하지 않음")
    labels = fitted_model.predict(values)
    return list(zip(frame["row_id"].tolist(), labels.tolist()))

A = pd.DataFrame([("A08", 8, 4), ("A02", 1, 2), ("A05", 4, np.nan)],
                 columns=["row_id", "texture", "color"])
B = pd.DataFrame([("B09", 9, 4), ("B04", 1, 1)],
                 columns=["row_id", "texture", "color"])
order_A = ["A05", "A08", "A02"]
order_B = ["B04", "B09"]
pairs_A = predict_pairs(model, A)
pairs_B = predict_pairs(model, B)
print("입력 A 행과 예측:", pairs_A)
print("출력해야 하는 A ID 순서:", order_A)


입력 A 행과 예측: [('A08', 1), ('A02', -1), ('A05', -1)]
출력해야 하는 A ID 순서: ['A05', 'A08', 'A02']


In [13]:
def align_predictions(expected_ids, pairs):
    ids = list(expected_ids)
    pairs = list(pairs)
    if any(is_missing_id(row_id) for row_id in ids):
        raise ValueError("요구 ID 결측")
    if len(ids) != len(set(ids)):
        raise ValueError("요구 ID 중복")
    mapping = {}
    for row_id, prediction in pairs:
        if is_missing_id(row_id):
            raise ValueError("예측 ID 결측")
        if row_id in mapping:
            raise ValueError("예측 ID 중복")
        if isinstance(prediction, (bool, np.bool_)) or not isinstance(prediction, (int, np.integer)):
            raise ValueError("예측은 정수 -1 또는 1")
        if int(prediction) not in (-1, 1):
            raise ValueError("예측은 -1 또는 1")
        mapping[row_id] = int(prediction)
    if set(mapping) != set(ids):
        raise ValueError("요구 ID와 예측 ID가 다름")
    return [mapping[row_id] for row_id in ids]

ordered_A = align_predictions(order_A, pairs_A)
ordered_B = align_predictions(order_B, pairs_B)
naive_A = [label for _, label in pairs_A]
print("A 입력 순서만 그대로 쓴 잘못된 값:", naive_A)
print("A 요구 ID 순서로 재배열한 값:", ordered_A)
print("B 요구 ID 순서로 재배열한 값:", ordered_B)
assert ordered_A == [-1, 1, -1]
assert ordered_B == [-1, 1]
assert naive_A != ordered_A


A 입력 순서만 그대로 쓴 잘못된 값: [1, -1, -1]
A 요구 ID 순서로 재배열한 값: [-1, 1, -1]
B 요구 ID 순서로 재배열한 값: [-1, 1]


### 두 CSV를 담은 ZIP 확인하기

조건을 하나씩 검사해 봅시다. 두 파일의 이름이 맞아도 행이 빠질 수 있고, 줄 수가 맞아도 A05의 자리에 A08의 예측이 들어갈 수 있습니다. ZIP을 다시 읽을 때는 예측을 만든 순간의 <strong>ID와 값의 짝</strong>에 대조합니다. 같은 값을 받은 두 작품을 맞바꾸면 CSV의 숫자만으로는 구별할 수 없습니다.


In [14]:
import csv
import io
import zipfile

def csv_text(labels):
    stream = io.StringIO(newline="")
    writer = csv.writer(stream, lineterminator="\n")
    writer.writerows([[value] for value in labels])
    return stream.getvalue()

def archive_texts(text_A, text_B):
    stream = io.BytesIO()
    with zipfile.ZipFile(stream, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("submissionA.csv", text_A)
        archive.writestr("submissionB.csv", text_B)
    return stream.getvalue()

def make_archive(ids_A, preds_A, ids_B, preds_B):
    a = align_predictions(ids_A, preds_A)
    b = align_predictions(ids_B, preds_B)
    return archive_texts(csv_text(a), csv_text(b))

def inspect_archive(blob, ids_A, preds_A, ids_B, preds_B):
    wanted = {"submissionA.csv": align_predictions(ids_A, preds_A),
              "submissionB.csv": align_predictions(ids_B, preds_B)}
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        names = archive.namelist()
        if len(names) != 2 or set(names) != set(wanted):
            raise ValueError("ZIP 파일명이 다르거나 중복됨")
        actual = {}
        for name, expected in wanted.items():
            rows = list(csv.reader(io.StringIO(archive.read(name).decode("utf-8"))))
            if any(len(row) != 1 or row[0] not in ("-1", "1") for row in rows):
                raise ValueError("헤더 없는 한 열의 -1 또는 1이 필요함")
            labels = [int(row[0]) for row in rows]
            if len(labels) != len(expected):
                raise ValueError("입력 행 수와 예측 행 수가 다름")
            if labels != expected:
                raise ValueError("저장한 ID 대응 순서가 다름")
            actual[name] = labels
    return actual

archive_bytes = make_archive(order_A, pairs_A, order_B, pairs_B)


In [15]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory(prefix="toy-submission-") as directory:
    practice_zip = Path(directory) / "submission.zip"
    practice_zip.write_bytes(archive_bytes)
    read_back = practice_zip.read_bytes()
    verified = inspect_archive(read_back, order_A, pairs_A, order_B, pairs_B)
    print("임시 작업 폴더에 저장한 파일:", practice_zip.name)
    print("파일을 다시 열어 확인한 두 CSV:", verified)
    with zipfile.ZipFile(practice_zip) as archive:
        print("A CSV 세 줄:", archive.read("submissionA.csv").decode("utf-8").splitlines())
    assert read_back == archive_bytes
# 블록이 끝나면 임시 파일은 삭제됩니다.
print("임시 파일 정리:", not practice_zip.exists())
assert not practice_zip.exists()


임시 작업 폴더에 저장한 파일: submission.zip
파일을 다시 열어 확인한 두 CSV: {'submissionA.csv': [-1, 1, -1], 'submissionB.csv': [-1, 1]}
A CSV 세 줄: ['-1', '1', '-1']
임시 파일 정리: True


임시 폴더에 쓴 <code>submission.zip</code>은 다시 열어 확인한 뒤 지웠습니다. 다음 중 어떤 실수는 줄 수만 세어도 찾을 수 있고, 어떤 실수는 예측 ID 기록을 함께 봐야 찾을 수 있을까요? 아래 검사는 조건을 어긴 경우의 오류를 보여 줍니다.


In [16]:
def show_rejected(name, action):
    try:
        action()
    except ValueError as error:
        print(name, "→", type(error).__name__, ":", error)
    else:
        raise AssertionError(name + " 오류를 놓쳤습니다")

show_rejected("필수 열 누락", lambda: predict_pairs(model, A.drop(columns=["color"])))
show_rejected("요구 ID 결측", lambda: align_predictions([None], [(None, 1)]))
show_rejected("예측 ID 결측", lambda: align_predictions(["A"], [(np.nan, 1)]))
show_rejected("예측 ID 중복", lambda: align_predictions(order_A, pairs_A + [pairs_A[0]]))
show_rejected("미상 0을 출력", lambda: align_predictions(order_A, [("A05", 0), ("A08", 1), ("A02", -1)]))
wrong_order_zip = archive_texts(csv_text(naive_A), csv_text(ordered_B))
show_rejected("같은 개수의 순서 오류",
              lambda: inspect_archive(wrong_order_zip, order_A, pairs_A, order_B, pairs_B))
header_zip = archive_texts("label\n" + csv_text(ordered_A), csv_text(ordered_B))
show_rejected("헤더 추가",
              lambda: inspect_archive(header_zip, order_A, pairs_A, order_B, pairs_B))
short_zip = archive_texts(csv_text(ordered_A[:-1]), csv_text(ordered_B))
show_rejected("예측 행 누락",
              lambda: inspect_archive(short_zip, order_A, pairs_A, order_B, pairs_B))


필수 열 누락 → ValueError : 필수 열 없음: color
요구 ID 결측 → ValueError : 요구 ID 결측
예측 ID 결측 → ValueError : 예측 ID 결측
예측 ID 중복 → ValueError : 예측 ID 중복
미상 0을 출력 → ValueError : 예측은 -1 또는 1
같은 개수의 순서 오류 → ValueError : 저장한 ID 대응 순서가 다름
헤더 추가 → ValueError : 헤더 없는 한 열의 -1 또는 1이 필요함
예측 행 누락 → ValueError : 입력 행 수와 예측 행 수가 다름


### 미상 표시를 다른 도구로 옮기면

Antique에서는 복제품이 −1이고 미상이 0입니다. scikit-learn의 반지도 학습 도구는 −1을 미상으로 읽습니다. 그대로 넣으면 어떤 확인된 그림을 놓치고, 어느 그림을 확인된 무리로 잘못 넣을까요? 뜻을 유지하는 한 가지 대응은 **복제품 −1→0, 진품 1→1, 미상 0→−1**입니다. 제출할 때는 예측한 0을 −1로, 1을 1로 되돌립니다. 이 대응은 기호를 맞추는 일이고, 미상 그림의 측정값에서 어떻게 배울지는 따로 풀어야 합니다. [scikit-learn의 미상 표시](https://scikit-learn.org/stable/modules/semi_supervised.html)

국내 중등부의 준비 범위는 KOAI 안내가 정하며, 위 그림 문제는 과거 국제 개인전입니다. 현재 연결된 출제 기준의 표제는 IOAI 2026입니다. [KOAI 2027 안내](https://kitpa.org/koai/2027) · [공식 출제 기준](https://kitpa.org/koai/syllabus)
